 * pure Python
 * implement decision tree and regression tree with Gini index, max depth, min number of data points
 * comparison with sklearn
 * visualization of the decision path
 * analysis of hyperparameters impact in sklearn
 * linear trees    

## Set up of data

We're taking the classic Titanic dataset

We did have to clean the data set by removing all unique columns

In [4]:
import numpy as np
import pandas as pd
import math
titanic = pd.read_csv('Titanic-Dataset.csv')

titanic.head()

# remove unique columns
titanic = titanic.drop(["PassengerId", "Name", "Cabin", "Ticket"], axis =1)

# change string columns to numbers
sex = {"female": 1, "male": 0}
embark = {"S": 0, "C": 1, "Q": -1}

titanic["Sex"] = titanic["Sex"].map(sex)
titanic["Embarked"] = titanic["Embarked"].map(embark)

# changing the name

data = titanic

data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0.0
1,1,1,1,38.0,1,0,71.2833,1.0
2,1,3,1,26.0,0,0,7.9250,0.0
3,1,1,1,35.0,1,0,53.1000,0.0
4,0,3,0,35.0,0,0,8.0500,0.0


We will now set up the features vs the targets.

In [5]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

feature = data[features]
target = data["Survived"]

feature.head()

# split into training and testing 80/20?

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,22.0,1,0,7.2500,0.0
1,1,1,38.0,1,0,71.2833,1.0
2,3,1,26.0,0,0,7.9250,0.0
3,1,1,35.0,1,0,53.1000,0.0
4,3,0,35.0,0,0,8.0500,0.0


## Gini index

$Gini=1−\sum_{i=1}^n ​p_i^2$

$Entropy=−\sum_{i=1}^n ​p_i\log_2(p_i)$

![alt text](images/image.png)

In [6]:
def gini(target):
    # split the target into survied and perished and count them
    elements, count = np.unique(target, return_counts=True)

    gini = 0
    total = len(target)
    
    for i in range(len(elements)):
        prob = count[i]/total
        gini += prob**2
    
    return 1-gini

def entropy(col):
    # calculate the unique elements and count them
    elements, count = np.unique(col, return_counts=True)

    entropy = 0
    total = len(col)

    for i in range(len(elements)):
        prob = count[i]/total
        entropy += prob * math.log2(prob)
    
    return - entropy


Dec tree

In [ ]:
def weighted_gini_cat(col: str, target: str, data: pd.DataFrame):
    # col = name of columnn you might want to split on
    # target = the target: here "survied"
    # data = the entire data set

    # catagorical values in the target column
    catagories = np.unique(data[col])

    weight = 0

    # catagory with lowest gini index [cat, gini]
    min_cat = [0 , 100.0]
               
    for cat in catagories:
        # for each catagory
        cat_target = list()

        for x in range(len(data[col])):
            if data[col][x] == cat:
                cat_target.append(data[target][x])

        # calculate the weighted gini (gini * instances/total length)
        weight_gini = gini(np.array(cat_target)) * (len(cat_target)/len(data[target])) 
        
        if weight_gini < min_cat[1]:
            min_cat[0] = cat
            min_cat[1] = weight_gini
        
        weight += weight_gini 

    return weight, min_cat[0]

def weighted_gini_num(col: str, target: str, data: pd.DataFrame):
    # col = name of columnn you might want to split on
    # target = the target: here "survied"
    # data = the entire data set

    # sort by values
    data.sort_values(col)

    # find the averages
    averages = list()

    for i in range(len(data[col])-1):
        averages.append((data[col][i]+data[col][i+1])/2)

    # average with lowest gini index [avg, gini]
    min_avg = [0 , 100.0]


    for avg in averages:
        # for each catagory
        avg_target = list()
    
        for x in range(len(data[col])):
            if data[col][x] > avg:
                avg_target.append(data[target][x])
    
        # calculate the weighted gini (gini * instances/total length)
        weight_gini = gini(np.array(avg_target)) * (len(avg_target)/len(data[target])) 

        if weight_gini < min_avg[1]:
            min_avg[0] = avg
            min_avg[1] = weight_gini
        
        weight += weight_gini
    
    return weight, min_avg[0]

In [ ]:
def find_split(data: pd.DataFrame, features: list, cat: list, target: str):
    smallest = {}

    # go through all features to find the one with the lowest gini
    for feat in features:
        if feat in cat:
            gini, condition = weighted_gini_cat(feat, target, data)
        else:
            gini, condition = weighted_gini_num(feat, target, data)

        if smallest:
            if smallest["gini"]> gini:
                smallest = {"feat": feat, "gini": gini, "condition": condition}
        else: 
            smallest = {"feat": feat, "gini": gini, "condition": condition}

    return smallest

def data_split(data : pd.DataFrame, feature: str, cat:bool, condition):
    # returns 2 data frames: one that satifies the condition and the other that doesnt
    # child data1 and child data2
    child_data1 = None
    child_data2 = None
    if cat:
        child_data1 = data[data[feature]== condition]
        child_data2 = data[data[feature]!= condition]

    else:
        child_data1 = data[data[feature] > condition]
        child_data2 = data[data[feature] <= condition]
    return child_data1, child_data2

In [ ]:
class Graph_Node:
    def __init__(self):
        self.leaf = False

    def is_leaf(self):
        return self.leaf

class Node(Graph_Node):
    def __init__(self, cat: bool, condition, gini):
        # if its catigorical, split condition and the gini index of that split
        self.cat = cat
        self.condition = condition
        self.gini = gini

    child1 = None
    child2 = None

    def set_children(self, child1: Graph_Node, child2: Graph_Node):
        self.child1 = child1
        self.child2 = child2

    def get_child1(self):
        return self.child1
    
    def get_child2(self):
        return self.child2

    def get_if_split(self, x):
        # returns true for child 1 and false for child 2
        if self.cat:
            return x == self.condition
        else:
            return x > self.condition

class Leaf(Graph_Node):
    def __init__(self, target_value):
        self.target_value = target_value
        self.leaf = True

    def resolve(self):
        


In [ ]:
class Tree:
    def __init__(self, data: pd.DataFrame, features: list, cat: list, target: str, max_depth : int):
        self.data = data
        self.features = features
        self.cat = cat
        self.target = target   
        self.max_depth = max_depth 

    current_depth = 0
    root : Node

    def get_root(self):
        return self.root

    def build_tree(self):
        # build first node by finding the best split
        split = find_split(self.data, self.features, self.cat, self.target)
        # {"feat": feat, "gini": gini, "condition": condition}

        self.root = Node(split["feat"] in self.cat, split["condition"], split["gini"])

        child_data_1, child_data_2 = data_split(self.data, split["feat"], split["feat"] in self.cat,split["condition"])


        self.current_depth +=1

        # build the other nodes in while loop till max depth
        self.root.set_children(self.build_nodes(child_data_1),
                               self.build_nodes(child_data_2))

    def build_nodes(self, data):
        # check end conditions
        if (self.current_depth >= self.max_depth or len(pd.unique(target)) == 1):
            return Leaf()

        split = find_split(data, self.features, self.cat, self.target)
        # {"feat": feat, "gini": gini, "condition": condition}

        parent = Node(split["feat"] in self.cat, split["condition"], split["gini"])

        child_data_1, child_data_2 = data_split(data, split["feat"], split["feat"] in self.cat,split["condition"])


        self.current_depth +=1

        # build the other nodes in while loop till max depth
        parent.set_children(self.build_nodes(child_data_1),
                               self.build_nodes(child_data_2))

        return parent
        

    def predict(self):
        # go through nodes
        pass

Regression Tree